# The fold

Section 1.2 of the lecture notes, at the keyboard.  An exchange publishes a stream of messages
and, separately, its own aggregation of them.  Here we fold the first, and practise the habit
the rest of the course is built on: compute one number by two routes and read the disagreement.

Two routes that must agree are a check.  Two routes that must *not* agree are a trap, and the
second half of this notebook is about one of those: a statistic that stays in range, moves with
the market, and answers a different question from the one asked.

Nothing before section 8 needs a data file.  A LOBSTER pair is five rows of text, and we write
one.

In [1]:
import time
from pathlib import Path
from tempfile import mkdtemp

import numpy as np
import pandas as pd

from unito26.lob import config, frames, lobster
from unito26.lob.lobster import LobsterEvent
from unito26.lob.lobster_session import (
    LobsterMarketSession, coarsening_report, coarsening_totals,
)
from unito26.lob.messages import (
    BUY, SELL, GridDepth, MessageType, ReportedDepth, SweepSize, TickGrid,
    limit_order, market_order, withdrawal,
)
from unito26.lob.orderbook import AggregateBook
from unito26.lob.session import MarketSession, run
from unito26.lob.simulate import OrderFlowSimulator
from unito26.lob.statistics import SessionStatistics
from unito26.lob.visualization import imbalance_figure, use_template
from unito26.lob.worked_examples import INDEXING_ASKS, INDEXING_BIDS

use_template()

GRID = TickGrid(0.01)
UNIT = lobster.price_unit(GRID)
SPEC = SessionStatistics((GridDepth(1), GridDepth(2)), (SweepSize(100),), (1,))
WHOLE_DAY = lobster.NASDAQ_REGULAR_HOURS

## 1. The two files

LOBSTER ships a *message* file and an *orderbook* file.  Row $k$ of the second is the
configuration after the event on row $k$ of the first, and that is the whole of the alignment:
the orderbook file carries no clock, no sequence number and no key.

`write_pair` is the inverse of the loader, so we can make a pair by hand and read it back.  The
one below is five messages: a submission, then one aggressive sell taken out of three resting
buy orders at the same price, then another submission.

In [2]:
PAIR_DEPTH = ReportedDepth(2)
NAME = "TICK_2012-06-21_34200000_57600000_{kind}_2.csv"


def pair(directory, messages, book):
    '''Write a hand-made LOBSTER pair, and read it back as the files hold it.'''
    Path(directory).mkdir(parents=True, exist_ok=True)
    files = lobster.LobsterFiles.parse(Path(directory) / NAME.format(kind="message"))
    lobster.write_pair(
        files,
        pd.DataFrame(messages, columns=frames.lobster_message_file_columns()),
        pd.DataFrame(book, columns=frames.lobster_book_columns(PAIR_DEPTH)),
    )
    return LobsterMarketSession.from_files(files, SPEC, GRID, WHOLE_DAY)


def state(bid_size, bid=100000, ask=100200, ask_size=120, second=99900, second_size=10):
    return [ask, ask_size, bid, bid_size, frames.ASK_PADDING, 0, second, second_size]


QUEUE_SPLIT = (
    [
        (34200.0, 1, 1, 10, 99900, 1),
        (34201.5, 4, 11, 20, 100000, 1),
        (34201.5, 4, 12, 20, 100000, 1),
        (34201.5, 4, 13, 15, 100000, 1),
        (34202.0, 1, 2, 10, 99900, 1),
    ],
    [state(100), state(80), state(60), state(45), state(45, second_size=20)],
)

workspace = Path(mkdtemp())
split = pair(workspace / "split", QUEUE_SPLIT[0], QUEUE_SPLIT[1])

print(split.messages.to_string())

      Time  TimeNanoseconds  Type  OrderID  Size   Price  Direction
0  34200.0   34200000000000     1        1    10   99900          1
1  34201.5   34201500000000     4       11    20  100000          1
2  34201.5   34201500000000     4       12    20  100000          1
3  34201.5   34201500000000     4       13    15  100000          1
4  34202.0   34202000000000     1        2    10   99900          1


In [3]:
print(split.book.to_string())

   AskPrice1  AskSize1  BidPrice1  BidSize1   AskPrice2  AskSize2  BidPrice2  BidSize2
0     100200       120     100000       100  9999999999         0      99900        10
1     100200       120     100000        80  9999999999         0      99900        10
2     100200       120     100000        60  9999999999         0      99900        10
3     100200       120     100000        45  9999999999         0      99900        10
4     100200       120     100000        45  9999999999         0      99900        20


The message frame carries the clock as a column; the book frame is indexed by position.  That
is not a presentational choice — a frame of one shape is refused by the other's schema, so the
alignment cannot quietly become a join on time.

Three of the five rows share the timestamp 34201.5 and the price 100000.  They are one sell
order eating three resting buy orders, and the file writes one row per resting order consumed.
Section 7 is about what that costs.

## 2. A message is a submission or a withdrawal

LOBSTER's `Type` column has seven documented values.  The book's own vocabulary has two.

In [4]:
print(f"{'LobsterEvent':<24} {'value':>5}   effect on the aggregated book")
print("-" * 78)
EFFECT = {
    LobsterEvent.SUBMISSION: "SUBMIT, at the stated price and side",
    LobsterEvent.PARTIAL_CANCELLATION: "WITHDRAW, the stated size",
    LobsterEvent.DELETION: "WITHDRAW, the whole resting order",
    LobsterEvent.EXECUTION_VISIBLE: "none of its own: the market part of another submission",
    LobsterEvent.EXECUTION_HIDDEN: "none: the size never rested in the visible book",
    LobsterEvent.CROSS_TRADE: "none: an auction print, outside continuous trading",
    LobsterEvent.TRADING_HALT: "none: a state change of the session, not of the book",
}
for event, effect in EFFECT.items():
    print(f"{event.name:<24} {int(event):>5}   {effect}")

print()
print("MessageType has", len(MessageType), "members:", [m.name for m in MessageType])

LobsterEvent             value   effect on the aggregated book
------------------------------------------------------------------------------
SUBMISSION                   1   SUBMIT, at the stated price and side
PARTIAL_CANCELLATION         2   WITHDRAW, the stated size
DELETION                     3   WITHDRAW, the whole resting order
EXECUTION_VISIBLE            4   none of its own: the market part of another submission
EXECUTION_HIDDEN             5   none: the size never rested in the visible book
CROSS_TRADE                  6   none: an auction print, outside continuous trading
TRADING_HALT                 7   none: a state change of the session, not of the book

MessageType has 2 members: ['SUBMIT', 'WITHDRAW']


Four of the seven move the book, and the book's vocabulary has two words for them.  It has two
because the aggregated state cannot tell a type 4 from a type 3: a level shrinks by cancellation
as well as by execution, and the two leave the same book behind.  Read the other way round, that
is why the volume traded is not recoverable from the states, and why a session carries its
trades beside its statistics rather than differencing them out of the sizes.

The aggressive order has no row of its own.  A marketable order never rests, so it is never a
type 1; its only trace is the executions it caused on the *resting* side, and the direction
those carry is the resting side's.  Section 8 checks that as an equality rather than as a sign.

Nothing in the package maps a `LobsterEvent` onto a `MessageType`.  That translation is where a
reconstruction of the book from the message file would begin, and it is left unwritten here on
purpose: the pair already carries the fold's input and its output, and the work of this notebook
is to compare them rather than to redo one of them.

## 3. The fold

$\mathcal{B}_k = F(\mathcal{B}_{k-1}, m_k)$, with $\mathcal{B}_0 = \varnothing$.  The book is
the accumulator: it holds one state, the current one, and the sequence of states belongs to
whatever drives the fold.

In [5]:
STREAM = [
    limit_order(1.0, 100, 1000, BUY),
    limit_order(2.0, 80, 1002, SELL),
    market_order(3.0, 40, SELL),
    withdrawal(4.0, 30, 1002, SELL),
]

book = AggregateBook()
for k, message in enumerate(STREAM, start=1):
    book.apply(message, record=False)
    print(f"B_{k}: bid {dict(book.levels_map(BUY))}  ask {dict(book.levels_map(SELL))}")

B_1: bid {1000: 100}  ask {}
B_2: bid {1000: 100}  ask {1002: 80}
B_3: bid {1000: 60}  ask {1002: 80}
B_4: bid {1000: 60}  ask {1002: 50}


The loop above *is* the fold, and the states printed are the driver's, not the book's: the book
holds only the last of them.  A driver that wanted one state per second, or one per evaluation
of a statistic, would print a different sequence from the same accumulator.

`run` is the same fold with no driver at all.  It keeps counts and throws the states away,
which is what a benchmark wants and what the next notebook uses.

In [6]:
print(run(AggregateBook(), STREAM))

Counter({'traded': 40, 'messages': 4, 'fills': 1, 'withdrawals': 1})


## 4. Folding a session

A stream long enough to have statistics.  The order flow comes from the Hawkes simulator of
section 1.3, which is ahead of us; here it is only a source of messages that read the book they
are folded into.

In [7]:
def simulated(online_statistics):
    '''A fresh simulator, a fresh book, and the same stream every time.'''
    simulator = OrderFlowSimulator(
        config.example_order_flow_params(), config.example_mark_params(),
        reference_price=10000, rng=7,
    )
    book = AggregateBook()
    simulator.warm_up(book, 30.0, None)
    return MarketSession.from_occupied_levels(
        book, simulator.stream(book, 120.0, None),
        ReportedDepth(5), SPEC, UNIT, online_statistics,
    )


session = simulated(True)
print(f"{len(session.lobster_book):,} messages folded, "
      f"{session.lobster_book.shape[1]} book columns, "
      f"{session.stats.shape[1]} statistics")
print()
print(session.stats[["Spread", "MidPrice", "QueueImbalance1", "OrderFlowContribution"]].head())

2,700 messages folded, 20 book columns, 28 statistics

           Spread  MidPrice  QueueImbalance1  OrderFlowContribution
TimeStamp                                                          
30.108556     7.0    9936.5         0.266667                    NaN
30.116979     7.0    9936.5         0.266667                    0.0
30.138095     5.0    9935.5         0.727273                  -30.0
30.144472     5.0    9935.5         0.727273                    0.0
30.159592     5.0    9935.5         0.727273                    0.0


The warm-up matters.  A book that starts empty has nothing to trade against, and its first
few hundred messages are a transient rather than a market; `warm_up` runs the same stream into
the book and discards the messages, so the session begins from a book that has been running.

## 5. Two routes to one number, and they agree

The fold writes the statistics as it goes, from a live book it is walking anyway.  The finished
frame determines them too, and `stats_from_frame` recomputes every one of them from the frame
alone, touching no book.

Two implementations, no shared arithmetic, one answer.

In [8]:
recomputed = session.stats_from_frame()

agree = [name for name in session.stats.columns if session.stats[name].equals(recomputed[name])]
print(f"{len(agree)} of {len(session.stats.columns)} columns identical")
print("differing:", [n for n in session.stats.columns if n not in agree])

28 of 28 columns identical
differing: []


In [9]:
start = time.perf_counter()
online = simulated(True)
with_statistics = time.perf_counter() - start

start = time.perf_counter()
bare = simulated(False)
without = time.perf_counter() - start

start = time.perf_counter()
bare.stats_from_frame()
afterwards = time.perf_counter() - start

print(f"fold with the statistics on : {with_statistics:.3f} s")
print(f"fold with them off          : {without:.3f} s")
print(f"then from the frame         : {afterwards:.3f} s")

fold with the statistics on : 0.231 s
fold with them off          : 0.144 s
then from the frame         : 0.021 s


The two routes cost different amounts, and which is cheaper is not the point.  The point is
that a quantity computed twice by unrelated code is a quantity whose value is evidence about
itself: a dropped message, an off-by-one in the write position, a statistic written before the
book was updated rather than after — each of those breaks one route and not the other.

A single route returns a number in range whatever it does.

## 6. Two routes that must not agree

The queue imbalance $I^n$ sums size over the first $n$ positions of the *price grid*.  A
LOBSTER file reports the first $L$ *occupied* prices.  Summing the first $n$ size columns of
the file is therefore a different sum, and the difference is invisible in the output.

The reference book of the notes has a hole on the bid side: 99 and 90 are occupied, and
everything between them is not.

In [10]:
reference = AggregateBook.from_levels(INDEXING_BIDS, INDEXING_ASKS)
print("bids:", INDEXING_BIDS)
print("asks:", INDEXING_ASKS)

row = reference.to_lobster_row(UNIT, ReportedDepth(2))
by_column = (row[3] + row[7] - row[1] - row[5]) / (row[3] + row[7] + row[1] + row[5])

print()
print("I^2 over the grid    :", round(reference.queue_imbalance(GridDepth(2)), 6))
print("I^2 over the columns :", round(by_column, 6))

bids: {99: 40, 90: 60}
asks: {101: 10, 102: 20, 103: 30, 104: 40, 105: 50}

I^2 over the grid    : 0.142857
I^2 over the columns : 0.538462


Both are well-formed imbalances.  Both lie in $[-1,1]$, both are positive, both say the bid is
heavier.  They differ by a factor of nearly four, because the second column of the file names a
price nine ticks from the touch and the second grid position names the price one tick from it,
where nothing rests.

At the scale of a session the difference is not an occasional artefact.

In [11]:
grid_imbalance = session.stats["QueueImbalance2"]
column_imbalance = session.column_sliced_imbalance(GridDepth(2))
gap = (grid_imbalance - column_imbalance).abs()

print(f"rows                 : {len(grid_imbalance):,}")
print(f"rows differing       : {int((gap > 1e-12).sum()):,}")
print(f"largest difference   : {gap.max():.4f}")
print(f"correlation          : {grid_imbalance.corr(column_imbalance):.4f}")
print(f"both inside [-1, 1]  : {column_imbalance.dropna().between(-1, 1).all()}")

rows                 : 2,700
rows differing       : 1,140
largest difference   : 1.2424
correlation          : 0.8548
both inside [-1, 1]  : True


In [12]:
imbalance_figure(session, GridDepth(2), slice(0, 400))

A correlation of that size is what makes the trap expensive.  The wrong series tracks the right
one closely enough to pass every plot and every sanity check, and differs on the rows where the
book has a hole — which are not a random subset of the rows, since a hole at the touch is what a
thin book looks like.

Nothing about a column of numbers in $[-1,1]$ says which of the two it is.  The name of the
function that produced it is the only record.

## 7. One order, several rows

The feed writes one row per resting order consumed; a session has one row per order that
consumed them.  Crossing between the two is a coarsening, and the question is what it preserves.

Write $e_n$ for the order flow contribution of row $n$ and $S$ for the size at the touch.  Over
the fills of one aggressive order,

$$\sum_n e_n - e(\text{first}, \text{last}) = S_{\text{end}} - \sum_i S_{j_i},$$

whose right-hand side vanishes exactly when the order emptied the touch at most once.  A queue
split — several resting orders at one price — never empties it more than once, and a level walk
always does.

The pair of section 1 is a queue split.

In [13]:
coarse_split = split.coarsened(True)
print(coarsening_totals(split, coarse_split).to_string(index=False))

                   Quantity         Fine  Coarse
                       rows     5.000000     3.0
                     shares    55.000000    55.0
traded value in tick-shares 55000.000000 55000.0
              VWAP in ticks  1000.000000  1000.0
   mean size of what trades    18.333333    55.0


In [14]:
report = coarsening_report(split, coarse_split)
print(report[report["RowsDiffering"] > 0][
    ["Statistic", "Dependence", "Guaranteed", "RowsCompared", "RowsDiffering"]
].to_string(index=False))
print()
print("order flow contribution, fine  :",
      split.stats_from_frame()["OrderFlowContribution"].iloc[1:4].sum())
print("order flow contribution, coarse:",
      coarse_split.stats["OrderFlowContribution"].iloc[1])

    Statistic Dependence  Guaranteed  RowsCompared  RowsDiffering
AverageDepth1     WINDOW       False             3              2

order flow contribution, fine  : -55.0
order flow contribution, coarse: -55.0


Five rows became three.  Volume, traded value and the session VWAP are equal to the bit, the
fills of an order being additive and every fill belonging to exactly one order.  The order flow
contribution survived too: three decrements at one price sum to the single decrement.

What moved is `AverageDepth1`, a per-row average — the rows were regrouped, so an average over
them is a different average.  That is a change of question, not an error.

Now the same order walking a level instead of splitting a queue.

In [15]:
LEVEL_WALK = (
    [
        (34200.0, 1, 1, 10, 99900, 1),
        (34201.5, 4, 11, 100, 100000, 1),
        (34201.5, 4, 12, 30, 99900, 1),
    ],
    [
        state(100, second_size=200),
        [100200, 120, 99900, 200, frames.ASK_PADDING, 0, 99800, 10],
        [100200, 120, 99900, 170, frames.ASK_PADDING, 0, 99800, 10],
    ],
)

walk = pair(workspace / "walk", LEVEL_WALK[0], LEVEL_WALK[1])
coarse_walk = walk.coarsened(True)

print("order flow contribution, fine  :",
      walk.stats_from_frame()["OrderFlowContribution"].iloc[1:].sum())
print("order flow contribution, coarse:",
      coarse_walk.stats["OrderFlowContribution"].iloc[1])

order flow contribution, fine  : -130.0
order flow contribution, coarse: -100.0


130 shares against 100.  The contribution reads the touch alone, so an order that walks
registers only the queue it emptied first, and the 30 shares taken at the next price are
invisible to it.  The identity said this would happen, and said where: on the walks and nowhere
else.

That is worth stating as a rule for reading any such report.  A statistic of one configuration
survives, because the surviving row *is* one of the file's rows.  A window sum of an extensive
quantity survives, because the fills of an order share a timestamp and so a window holds them
whole or not at all.  A function of two consecutive states, or a per-row average, does not.

## 8. The real file, if it is here

Everything above ran on rows we wrote.  A sample file, when one is present, is what says the
account is right about the data rather than about the fixture.

In [16]:
LOBSTER_DIR = Path("../../data/lobster")
SAMPLE = LOBSTER_DIR / "AMZN_2012-06-21_34200000_57600000_message_10.csv"

if SAMPLE.exists():
    amzn = lobster.LobsterFiles.parse(SAMPLE)
    print(lobster.event_census([amzn]).to_string(index=False))
else:
    amzn = None
    print("no sample file under", LOBSTER_DIR.resolve())
    print("sections 1 to 7 stand; the counts below are the ones they cannot supply")

Ticker  Depth  Messages  SUBMISSION  PARTIAL_CANCELLATION  DELETION  EXECUTION_VISIBLE  EXECUTION_HIDDEN  CROSS_TRADE  TRADING_HALT
  AMZN     10    269748      131954                  2917    123458               8974              2445            0             0


No cross trade and no trading halt on this day, and the columns are there regardless: they come
from the format, not from the data, and a schema written from what a file happens to hold
inherits that file's blind spots.

Two claims of the notes are checkable directly.  A visible execution prints at the prior best
price on the side that *rested* — `executions_off_the_touch` returns the rows where it does not,
and price-time priority says there are none.  And a file pads a level only where the side has
fewer than $L$ occupied prices, which on a depth-10 sample of a liquid name never happens.

In [17]:
if amzn is not None:
    day = LobsterMarketSession.from_files(amzn, SPEC, GRID, WHOLE_DAY)
    print(f"{len(day.messages):,} messages")
    print("visible executions away from the resting side's best price:",
          len(day.executions_off_the_touch()))
    print("levels that pad anywhere in the session              :",
          len(lobster.padding_census(amzn, WHOLE_DAY)))

269,748 messages
visible executions away from the resting side's best price: 0


levels that pad anywhere in the session              : 0


Both empty, and both are the claim rather than an absence of one.

Now the measurement section 7 predicted.  Count the market orders, count the ones that walked a
level, and ask the report how many rows of the order flow contribution changed.

In [18]:
if amzn is not None:
    orders = day.market_orders()
    fills = np.bincount(orders[orders >= 0])
    price = day.messages["Price"].to_numpy()
    visible = (day.messages["Type"] == LobsterEvent.EXECUTION_VISIBLE).to_numpy()
    walked = sum(
        len(np.unique(price[(orders == order) & visible])) > 1
        for order in np.flatnonzero(fills > 1)
    )

    print(f"market orders        : {len(fills):,}")
    print(f"  of several fills   : {int((fills > 1).sum()):,}")
    print(f"  that walked a level: {walked:,}")

market orders        : 6,591
  of several fills   : 1,511
  that walked a level: 297


In [19]:
if amzn is not None:
    coarse_day = day.coarsened(True)
    day_report = coarsening_report(day, coarse_day)
    print(day_report[day_report["RowsDiffering"] > 0][
        ["Statistic", "Guaranteed", "RowsCompared", "RowsDiffering"]
    ].to_string(index=False))
    print()
    print(coarsening_totals(day, coarse_day).to_string(index=False))

            Statistic  Guaranteed  RowsCompared  RowsDiffering
OrderFlowContribution       False        264921            297
  OrderFlowImbalance1       False        264921          15630
        AverageDepth1       False        264921          70113

                   Quantity         Fine       Coarse
                       rows 2.697480e+05 2.649210e+05
                     shares 6.132480e+05 6.132480e+05
traded value in tick-shares 1.365336e+10 1.365336e+10
              VWAP in ticks 2.226401e+04 2.226401e+04
   mean size of what trades 6.833608e+01 9.304324e+01


The rows on which the order flow contribution differs are the level walks, counted
independently and to the row.  A proof of where two routes must agree, and a measurement that
lands on it.

The last line of the totals is the trap LOBSTER's own demonstration code names.  The mean of
the fine column is the mean *execution*; the mean of the coarse one is the mean *trade*; the
two differ by a third, and nothing in the file marks which one a given frame holds.

## 9. What to take away

The book is the accumulator of a fold.  It holds one state, the transition is one function, and
the sequence of states belongs to the driver rather than to the book.

Of LOBSTER's seven event types, two move the book.  An execution is somebody else's submission
read on the passive side, and applying it as an event of its own counts the trade twice.

Compute the things you care about twice.  Where the two routes must agree, the agreement is
evidence; where they must not, know which one you are holding, because both will be in range.

The granularity of a feed is part of its meaning.  One aggressive order is several rows, and a
statistic that reads two consecutive states cannot survive a regrouping of them — which is
provable in advance, and then measurable to the row.